# Zero-shot **DAD** — LaGoVAD (released ckpt) vs W0 (no SA²) vs R4 (SA²)

Rule (written before any DAD number): `core/docs/final/ZS_DAD.md`. Same three rows as `ZS_TAD_DADA.md` §7; the
read-out adds a DAD column to `REPORTS/zs_lagovad_readout.md`. Prerequisite: `zs_lagovad.ipynb` has run (DoTA / TAD /
DADA reads on Drive).

| step | what | GPU? |
|---|---|---|
| 1–3 | setup, deps, helpers | no |
| 4 | **census** (decode count, fps 25, annotation stats printed) + gates §3 → stop on any failure | no |
| 5 | labels **toa = 90** (last 10 frames of a positive): `Thesis-Final/data/DAD_s7` at s7, the arms' `defs.json` | no |
| 6 | decode mp4 → JPEG folders (VM disk) | no |
| 7 | CLIP `_ncc` s7 → `Thesis-Final/cache/clip/DAD_s7_ncc` | **yes** |
| 8 | VideoMAE V2-S s7, frame step 3 (16 × 3 = 1.8 s at 25 fps) → `Thesis-Final/cache/video/…/DAD_s7_squash` | **yes** |
| 9 | bake A2 with Nexar's statistics → `Thesis-Final/cache/v2/A2_S/DAD_s7_nexar` | no |
| 10 | SA² windows (1×1 + 7×7) → top-12 patches → `Thesis-Final/cache/{clip_windows,clip_patches}/` | **yes** |
| 11 | integrity §4 (plain == bake CLIP columns, rows, manifests, J10 + configs) | no |
| 12 | `core.evaluate` × 3 rows (LaGoVAD / W0 / R4) | yes (light) |
| 13 | `zs_lagovad_readout` → **paste `REPORTS/zs_lagovad_readout.md` to Claude** | no |

Every step skips what is already on Drive. One seed ⇒ **PRELIMINARY (n = 1 seed)**, adopts nothing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data/DAD, ckpts/best.ckpt
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 Nexar statistics, Nexar V2-S manifest, Nexar defs
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'     # every DAD cache lands under Thesis-Final/cache
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final-lagovad-zs there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'DAD_STRIDE'), 'this upload predates the DAD build -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/ZS_DAD.md').is_file(), 'no ZS_DAD.md (the rule) in this upload'

DSET     = constants.DAD_DATASET                                       # 'DAD'
STRIDE   = constants.DAD_STRIDE                                        # 7 (0.28 s at 25 fps)
VSTEP    = constants.DAD_VIDEOMAE_FRAME_STEP                           # 3 (16 x 3 = 1.8 s at 25 fps)
SEED     = constants.ZS_SEED
LAG, RUN_NAME = constants.ZS_LAGOVAD_ARM, constants.ZS_LAGOVAD_RUN
CONTROL, SA2 = constants.ZS_CONTROL_ARM, constants.ZS_SA2_ARM
ENC      = constants.V5_ENCODER                                        # V2-S
K, MOTION, CELLS = constants.STP_TOP_K, constants.STP_MOTION_RAW, constants.STP_CELLS_RESIZED
C        = constants.NEXAR_CONSTRUCTION_WINDOW
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV

DAD_ROOT = constants.DATA_ROOT / DSET                                  # Thesis/data/DAD (read-only)
CKPT     = Path(DRIVE) / 'ckpts' / 'best.ckpt'
DEFS     = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{C}' / constants.DEFS_FILENAME
A2_STATS = Path(DRIVE_V2) / 'cache' / 'v2' / 'A2_S' / 'Nexar_s8'       # W0's training bake: the Nexar statistics
NEXAR_V2S = Path(DRIVE_V2) / 'cache' / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'

CACHE    = constants.CACHE_ROOT                                        # Thesis-Final/cache
DATA     = Path(DRIVE_F) / 'data' / f'{DSET}_s{STRIDE}'                # labels + defs (core.evaluate data dir)
CLIP     = CACHE / 'clip' / f'{DSET}_s{STRIDE}_ncc'
V2S      = CACHE / 'video' / ENC / f'{DSET}_s{STRIDE}_squash'
BAKE     = CACHE / 'v2' / 'A2_S' / f'{DSET}_s{STRIDE}_nexar'

from core.data import stp_patches as stp_data   # noqa: E402
from core.tools import clip_windows             # noqa: E402

WIN      = clip_windows.cache_dir(DSET, STRIDE, constants.STP_WINDOW_GRIDS, CELLS)
PATCH    = stp_data.cache_dir(DSET, STRIDE, MOTION, K)
RUN_DIR  = {CONTROL: constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / CONTROL / f's{SEED}',
            SA2: constants.OUTPUT_ROOT / constants.STP_OUTPUT_DIRNAME / SA2 / f's{SEED}'}
ROOT     = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME   # zs_lagovad (the other three sets live here)
EVAL     = ROOT / 'eval' / constants.DAD_OUTPUT_SET
ZS_ROOT  = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME
ARM_DOTA = {CONTROL: constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / CONTROL,
            SA2: constants.OUTPUT_ROOT / constants.STP_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / SA2}
REPORTS  = ROOT / 'REPORTS'
CENSUS   = REPORTS / constants.DAD_CENSUS_FILENAME
LOCAL    = Path('/content/zs_dad')
FRAMES   = LOCAL / 'frames'                                            # VM disk, never Drive
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

for name, path in (('DAD videos', DAD_ROOT / 'videos' / constants.DAD_SPLIT_DIRNAME),
                   ('DAD annotation', DAD_ROOT / constants.DAD_ANNOTATION_DIRNAME), ('best.ckpt', CKPT),
                   ('Nexar defs', DEFS), ('A2 Nexar stats', A2_STATS), ('Nexar V2-S manifest', NEXAR_V2S / constants.VIDEO_MANIFEST_FILENAME),
                   *[(f'{a} run', RUN_DIR[a] / 'checkpoint_last.pt') for a in RUN_DIR],
                   ('zs_lagovad reads (run zs_lagovad.ipynb first)', ROOT / 'eval' / 'dada' / LAG)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:24s} {path}')
print('caches ->', CLIP, V2S, BAKE, WIN, PATCH, sep='\n  ')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 7, 8, 10 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst):
    """A whole Drive dir to VM disk once (bit-identical; C2)."""
    if not dst.exists():
        tmp = dst.with_name(dst.name + '.partial')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        os.replace(tmp, dst)
    return dst


def complete(d, ids):
    return d.is_dir() and all((d / f'{v}.npy').exists() for v in ids)


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def ckpt_step(path):
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 4. Census + gates (`ZS_DAD.md` §3) — stops here on any failure

Decodes every test video once (exact frame count) and measures fps (must be 25). The annotations are parsed and their
flag statistics printed only: the flag marks the involved objects from their first appearance, so labels follow the DAD
convention **toa = 90** (Amendment 1, `ZS_DAD.md` §7). A census from the first (20 fps) run is re-done automatically.

In [ ]:
def census_current(path):
    """A census written by this rule (toa labels, 25 fps); the first run's (flag / 20 fps) is redone."""
    if not path.exists():
        return False
    s = json.loads(path.read_text())['summary']
    return s['pass'] and 'label_rule' in s and s['fps'] == [float(constants.DAD_ASSUMED_FPS)]


if not census_current(CENSUS):
    local = LOCAL / constants.DAD_CENSUS_FILENAME
    try:
        run([sys.executable, '-m', 'core.tools.dad', 'census', '--root', DAD_ROOT, '--out', local],
            log=LOCAL / 'census.log')
    finally:
        if local.exists():
            publish(local, CENSUS)
doc = json.loads(CENSUS.read_text())
print(json.dumps(doc['summary'], indent=1))
assert doc['summary']['pass'], 'DAD census FAILED -- paste the summary above to Claude'
IDS = sorted(doc['videos'])
ID_FILE = write_ids(LOCAL / 'dad_ids.txt', IDS)
print(f"{len(IDS)} videos ({doc['summary']['positive']} positive / {doc['summary']['negative']} negative)")

## 5. Labels → `Thesis-Final/data/DAD_s7` (positive native[90:] = 1; rows = native[::7]; the arms' `defs.json`)

In [ ]:
if not (DATA / constants.DAD_MANIFEST_FILENAME).exists():
    run([sys.executable, '-m', 'core.tools.dad', 'prepare', '--census', CENSUS, '--defs', DEFS,
         '--stride', STRIDE, '--out-dir', LOCAL / 'data'])
    publish(LOCAL / 'data', DATA)
man = json.loads((DATA / constants.DAD_MANIFEST_FILENAME).read_text())
print(man)
assert man['stride'] == STRIDE and man['videos'] == len(IDS) and man['two_class_at_stride'] > 0, man
labels = json.loads((DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text())
assert sorted(labels) == IDS

## 6. Decode mp4 → JPEG folders (VM disk) — only if some cache below is incomplete

In [ ]:
NEED_FRAMES = not (complete(CLIP, IDS) and complete(V2S, IDS) and complete(WIN, IDS))
if NEED_FRAMES:
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.dad', 'decode', '--root', DAD_ROOT, '--census', CENSUS,
         '--out-dir', FRAMES], log=LOCAL / 'decode.log')
    print(f'decoded in {(time.time() - t0) / 60:.1f} min')
else:
    print('every frame-reading cache is complete -- not decoded')

## 7. CLIP `_ncc` at stride 7 → `Thesis-Final/cache/clip/DAD_s7_ncc`

In [ ]:
if complete(CLIP, IDS):
    print(f'{CLIP} complete')
else:
    assert torch.cuda.is_available(), 'no GPU'
    local = LOCAL / 'clip'
    run([sys.executable, '-m', 'core.tools.extract_clip_features', '--frames-dir', FRAMES, '--ids-file', ID_FILE,
         '--dataset', DSET, '--stride', STRIDE, '--batch-size', 64, '--device', 'auto', '--no-center-crop',
         '--output-dir', local], log=LOCAL / 'clip.log')
    publish(local, CLIP)
    (LOCAL / 'clip_marker.txt').write_text(f'transform=anisotropic_resize (--no-center-crop)\nstride={STRIDE}\nsource=dad decode JPEG q{constants.DAD_JPEG_QUALITY}\n')
    publish(LOCAL / 'clip_marker.txt', CLIP / 'TRANSFORM.txt')
    flush_drive()
bad = [v for v in IDS if rows(CLIP, v) != len(labels[v])]
assert not bad, f'CLIP rows != s{STRIDE} labels on {bad[:5]}'
print(f'CLIP OK: rows == labels on {len(IDS)} videos')

## 8. VideoMAE V2-S at stride 7, frame step 3 (1.8 s at 25 fps) → `Thesis-Final/cache/video/…/DAD_s7_squash`

In [ ]:
if complete(V2S, IDS):
    print(f'{V2S} complete')
else:
    assert torch.cuda.is_available(), 'no GPU'
    local, t0 = LOCAL / 'v2s', time.time()
    run([sys.executable, '-m', 'core.tools.extract_video_features', '--frames-dir', FRAMES, '--ids-file', ID_FILE,
         '--dataset', DSET, '--encoder', ENC, '--stride', STRIDE, '--clip-frame-step', VSTEP,
         '--assumed-fps', constants.DAD_ASSUMED_FPS, '--batch-size', 32, '--device', 'auto',
         '--output-dir', local], log=LOCAL / 'v2s.log')
    print(f'V2-S in {(time.time() - t0) / 60:.1f} min')
    publish(local, V2S)
    flush_drive()
mine = json.loads((V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
ref = json.loads((NEXAR_V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
diff = sorted(k for k in set(mine) | set(ref) if mine.get(k) != ref.get(k))
assert set(diff) <= {'stride', 'clip_frame_step', 'assumed_fps'}, f'DAD V2-S manifest differs from Nexar\'s on {diff}'
assert (mine['stride'], mine['clip_frame_step'], mine['assumed_fps']) == (STRIDE, VSTEP, constants.DAD_ASSUMED_FPS), mine
bad = [v for v in IDS if rows(V2S, v) != rows(CLIP, v)]
assert not bad, f'V2-S rows != CLIP rows on {bad[:5]} (C13)'
print(f'V2-S OK: manifest == Nexar\'s except {diff}; rows == CLIP rows')

## 9. Bake A2 with **Nexar's** statistics → `Thesis-Final/cache/v2/A2_S/DAD_s7_nexar` (CPU)

In [ ]:
man_path = BAKE / constants.V2_INPUT_MANIFEST_FILENAME
if not (man_path.exists() and json.loads(man_path.read_text())['baked_ids'] == len(IDS)):
    local = LOCAL / 'bake'
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', stage(A2_STATS, LOCAL / 'stats_a2'),
         '--clip-dir', CLIP, '--video-dir', V2S, '--ids-file', ID_FILE, '--stride', 1, '--out-dir', local],
        log=LOCAL / 'bake.log')
    publish(local, BAKE)
    flush_drive()
m = json.loads(man_path.read_text())
nexar_sha = json.loads((A2_STATS / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC) and m['train_ids_sha1'] == nexar_sha, m
bad = [v for v in IDS if rows(BAKE, v) != rows(CLIP, v)]
assert not bad, f'bake rows != CLIP rows on {bad[:5]}'
print(f'bake OK: A2 (no CRN, V2-S), Nexar statistics {nexar_sha[:12]}')

## 10. SA² patches: windows (1×1 + 7×7 of the 224² squash, 1×1 gated vs the CLIP rows) → top-12 by raw Mo

In [ ]:
if not complete(WIN, IDS):
    assert torch.cuda.is_available(), 'no GPU'
    local, t0 = LOCAL / 'win', time.time()
    if (WIN / constants.VIDEO_MANIFEST_FILENAME).exists():          # resume: keep what is on Drive
        stage(WIN, local)
    run([sys.executable, '-m', 'core.tools.clip_windows', 'frames', '--frames-dir', FRAMES, '--ids-file', ID_FILE,
         '--dataset', DSET, '--clip-dir', CLIP, '--stride', STRIDE, '--grids', 1, constants.STP_GRID,
         '--cells', CELLS, '--output-dir', local, '--device', 'auto'], log=LOCAL / 'windows.log')
    print(f'windows in {(time.time() - t0) / 60:.1f} min')
    publish(local, WIN)
    flush_drive()
if not complete(PATCH, IDS):
    local = LOCAL / 'patch'
    run([sys.executable, '-m', 'core.tools.stp_patches', 'select', '--windows-dir', WIN, '--ids-file', ID_FILE,
         '--cells', CELLS, '--motion', MOTION, '--top-k', K, '--out-dir', local,
         '--report', REPORTS / 'dad_select.json'])
    publish(local, PATCH)
    flush_drive()
bad = [v for v in IDS if np.load(PATCH / f'{v}.npy', mmap_mode='r').shape != (rows(BAKE, v), K * constants.STP_PATCH_DIM)]
assert not bad, f'patch rows != bake rows on {bad[:5]} (C13)'
print(f'patch cache OK: {len(IDS)} videos')

## 11. Integrity (`ZS_DAD.md` §4): LaGoVAD's plain rows == the bake's CLIP columns; J10 + configs

In [ ]:
L = {'clip': stage(CLIP, LOCAL / 'stage' / 'clip'), 'bake': stage(BAKE, LOCAL / 'stage' / 'bake'),
     'patch': stage(PATCH, LOCAL / 'stage' / 'patch'), 'data': stage(DATA, LOCAL / 'stage' / 'data')}
run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L['clip'], '--bake', L['bake'],
     '--ids-file', ID_FILE, '--plain-stride', 1])
WANT = {CONTROL: {'sa2': False}, SA2: {'sa2': True, 'motion': MOTION, 'top_k': K}}
for arm, d in RUN_DIR.items():
    step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
    assert last is not None and step == last['global_step'], f'{arm}: J10 (ckpt {step}, metrics {last and last["global_step"]})'
    cfg = yaml.safe_load((d / 'config.yaml').read_text())
    got = {k: cfg['stp'][k] for k in WANT[arm]}
    assert got == WANT[arm] and cfg['stp']['text_prompt'] == constants.STP_PROMPT_RELEASED, f'{arm}: stp config {cfg["stp"]}'
    print(f'{arm} s{SEED}: step {step} (J10 OK), stp {got}, prompts released')

## 12. Score: LaGoVAD / W0 / R4 with `core.evaluate` (whole video, s7, raw scores, `--save-scores`)

In [ ]:
common = ['--set', f'data.dataset={DSET}', '--data-dir', L['data'], '--score-norm', 'auto', '--save-scores']
JOBS = {
    LAG: (RUN_NAME, ['--baseline-ckpt', CKPT, *[x for o in constants.RELEASED_CKPT_OVERRIDES for x in ('--set', o)],
                     '--clip-dir', L['clip']]),
    CONTROL: (f's{SEED}', ['--ckpt', RUN_DIR[CONTROL] / 'checkpoint_last.pt', '--clip-dir', L['bake']]),
    SA2: (f's{SEED}', ['--ckpt', RUN_DIR[SA2] / 'checkpoint_last.pt', '--clip-dir', L['bake'],
                       '--patch-dir', L['patch'], '--patch-stride', STRIDE]),
}
wrote = False
for row, (name, args) in JOBS.items():
    out = EVAL / row / name
    if (out / 'results.json').exists():
        print(f'dad {row}: already read')
        continue
    local = LOCAL / 'eval' / row / name
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.evaluate', *args, *common, '--output-dir', local], log=LOCAL / f'eval_{row}.log')
    publish(local, out)
    wrote = True
for row, (name, _) in JOBS.items():
    r = json.loads((EVAL / row / name / 'results.json').read_text())
    print(f'dad {row}: macro {r["auc_macro"]:.4f} over {r["auc_macro_videos"]} | micro {r["auc"]:.4f} | norm {r["score_norm"]}')
if wrote:
    flush_drive()

## 13. Read-out (all four sets) → **paste `REPORTS/zs_lagovad_readout.md` to Claude**

In [ ]:
run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', '--lagovad-root', ROOT, '--zs-root', ZS_ROOT,
     '--dota', CONTROL, ARM_DOTA[CONTROL], '--dota', SA2, ARM_DOTA[SA2], '--seed', SEED])
print((REPORTS / 'zs_lagovad_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/zs_lagovad_readout.md to Claude')